# 03 - Procesamiento de Imágenes y Análisis de Red

Este notebook convierte los screenshots de Google Maps en datos de tráfico por calle,
y combina los datos observacionales con las métricas topológicas de la red.

**Pipeline:**
1. Cargar imágenes y convertir a matrices de tráfico (0-255)
2. Extraer valores de tráfico por calle (con distintos radio/steps)
3. Generar CSVs de combinaciones (48 archivos)
4. Combinar datos observacionales + computacionales
5. Visualizar resultados del procesamiento

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import scienceplots

from src.image_analysis.GetDataFromImages import ImageProcessor

## 1. Configuración de parámetros de procesamiento

In [2]:
# Parámetros de muestreo
COMBINACIONES = [
    (0, 2),   # r0s3
    (0, 5),   # r0s6
    (0, 9),   # r0s10
    (1, 2),   # r1s3  (paper N1207)
    (1, 5),   # r1s6  (paper N505)
    (1, 9),   # r1s10
    (2, 2),   # r2s3
    (2, 5),   # r2s6
    (2, 9),   # r2s10
    (3, 2),   # r3s3
    (3, 5),   # r3s6
    (3, 9),   # r3s10
]

REDES = {
    'N505': {
        'screenshots':  'data/raw/Images/screenshotsGoogleMaps/screenshots',
        'network_data': 'data/network/dat_files/PuntaArenas',
    },
    'N1207': {
        'screenshots':  'data/raw/Images/screenshotsGoogleMaps/screenshots',
        'network_data': 'data/network/dat_files/PuntaArenasDetallado',
    },
}

print("Combinaciones a generar:")
print(f"  Redes: {list(REDES.keys())}")
print(f"  Radio x Steps: {len(COMBINACIONES)} combinaciones")
print(f"  Total: {len(REDES) * len(COMBINACIONES) * 2} archivos CSV (mean + max)")

Combinaciones a generar:
  Redes: ['N505', 'N1207']
  Radio x Steps: 12 combinaciones
  Total: 48 archivos CSV (mean + max)


## 2. Ejemplo: Procesar una imagen individual

In [3]:
# Crear processor para N505
processor_N505 = ImageProcessor(
    path_dir_screenshots  = REDES['N505']['screenshots'],
    path_dir_network_data = REDES['N505']['network_data'],
)

# Procesar UNA sola imagen para demostración
ejemplo_file = '2023-04-03_12-00.png'  # Martes al mediodía
path_ejemplo = os.path.join(REDES['N505']['screenshots'], ejemplo_file)
img = cv2.imread(path_ejemplo)

# Convertir a matriz de tráfico
traffic_matrix = processor_N505._image_to_traffic_matrix(img)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
axes[0].imshow(img_rgb)
axes[0].set_title('Imagen original')

axes[1].imshow(traffic_matrix, cmap='hot', vmin=0, vmax=255)
axes[1].set_title('Matriz de tráfico')

# Histograma de valores
vals = traffic_matrix[traffic_matrix > 0].flatten()
axes[2].hist(vals, bins=50, color='steelblue', edgecolor='black')
axes[2].set_xlabel('Valor de tráfico')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title(f'Distribución ({len(vals)} píxeles)')

for ax in axes[:2]:
    ax.axis('off')

plt.suptitle(f'Procesamiento de imagen - {ejemplo_file}', fontsize=14)
plt.tight_layout()
plt.savefig('results/figures/procesamiento_imagen_ejemplo.png', dpi=200, bbox_inches='tight')
plt.show()

[ WARN:0@7.549] global loadsave.cpp:278 findDecoder imread_('data/raw/Images/screenshotsGoogleMaps/screenshots/2023-03-29_07-30.png'): can't open/read file: check file path/integrity


FileNotFoundError: No se encontró la imagen de referencia: data/raw/Images/screenshotsGoogleMaps/screenshots/2023-03-29_07-30.png

## 3. Efecto del radio y steps en el muestreo

In [ ]:
# Comparar distintos valores de radio y steps para una calle específica
calle_ejemplo = 0  # Primera conexión
conn = processor_N505.connections[calle_ejemplo]
p1 = processor_N505.position_of_vertices[conn[0]]
p2 = processor_N505.position_of_vertices[conn[1]]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for ax, (radio, steps) in zip(axes.flat[:7], COMBINACIONES[:7]):
    coords = processor_N505._intermediate_coords(list(p1), list(p2), steps)
    
    # Dibujar imagen con puntos
    img_display = cv2.cvtColor(img.copy(), cv2.COLOR_BGR2RGB)
    for (col, row) in coords:
        for di in range(-radio, radio + 1):
            for dj in range(-radio, radio + 1):
                r, c = row + di, col + dj
                if 0 <= r < img_display.shape[0] and 0 <= c < img_display.shape[1]:
                    img_display[r, c] = [255, 255, 0]
    
    ax.imshow(img_display)
    ax.set_title(f'r={radio}, s={steps+1}', fontsize=11)
    ax.axis('off')

# Ocultar último eje
axes.flat[7].axis('off')

plt.suptitle(f'Efecto de radio/steps en muestreo - Calle {calle_ejemplo}', fontsize=14)
plt.tight_layout()
plt.savefig('results/figures/efecto_radio_steps.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Generar CSVs de combinaciones

Este paso procesa las ~2900 imágenes para cada combinación de red x radio x steps.
**Tiempo estimado: 3-5 horas.** Descomentar la celda siguiente para ejecutar.

In [ ]:
# ⚠️ DESCOMENTAR PARA EJECUTAR EL PROCESAMIENTO COMPLETO ⚠️
# Esto tomará varias horas.

# import time
# OUTPUT_DIR = 'data/processed/combinaciones'
# os.makedirs(OUTPUT_DIR, exist_ok=True)
#
# for red_nombre, red_config in REDES.items():
#     print(f"\n--- Red: {red_nombre} ---")
#     processor = ImageProcessor(
#         path_dir_screenshots  = red_config['screenshots'],
#         path_dir_network_data = red_config['network_data'],
#     )
#     for radio, steps in COMBINACIONES:
#         s_label = steps + 1
#         nombre = f"r{radio}s{s_label}"
#         path_mean = os.path.join(OUTPUT_DIR, f"traffic_mean_{red_nombre}_{nombre}.csv")
#         path_max  = os.path.join(OUTPUT_DIR, f"traffic_max_{red_nombre}_{nombre}.csv")
#         if os.path.exists(path_mean) and os.path.exists(path_max):
#             print(f"  [SKIP] {nombre} ya existe")
#             continue
#         print(f"  [START] {nombre}...", end=' ', flush=True)
#         t0 = time.time()
#         df_mean, df_max = processor.images_to_dataframe(steps=steps, radio=radio)
#         df_mean.to_csv(path_mean, index=False)
#         df_max.to_csv(path_max, index=False)
#         print(f"{time.time()-t0:.0f}s")
#
# print("Procesamiento completado.")

## 5. Cargar y explorar datos procesados

In [ ]:
# Cargar datos de combinaciones pre-procesadas
OUTPUT_DIR = 'data/processed/combinaciones'

def load_combinacion(red, radio, steps, tipo='mean'):
    s_label = steps + 1
    path = os.path.join(OUTPUT_DIR, f'traffic_{tipo}_{red}_r{radio}s{s_label}.csv')
    return pd.read_csv(path)

# Ejemplo: cargar N505 r1s6 (paper)
df = load_combinacion('N505', radio=1, steps=5, tipo='mean')
print(f"Forma: {df.shape}")
print(f"Columnas (primeras 5): {list(df.columns[:5])}")
print(f"Columnas (últimas 5): {list(df.columns[-5:])}")
display(df.head())

In [ ]:
# Estadísticas de tráfico
traffic_cols = [c for c in df.columns if c not in ['connection', 'lanes', 'length']]
traffic_data = df[traffic_cols]

print(f"Rango de valores: [{traffic_data.values.min():.0f}, {traffic_data.values.max():.0f}]")
print(f"Promedio global: {traffic_data.values.mean():.1f}")
print(f"Calles con datos (mean > 0): {(traffic_data.mean(axis=1) > 0).sum()} / {len(df)}")

## 6. Cargar datos de centralidad de la red

In [ ]:
# Datos computacionales (centralidad)
datos_comp_N505 = pd.read_csv('data/processed/DataNetwork/StreetAsNode/all_data_SimpleNet_new.csv')
datos_comp_N1207 = pd.read_csv('data/processed/DataNetwork/StreetAsNode/all_data_ComplexNet_new.csv')

print("N505 - Datos computacionales:")
print(datos_comp_N505.columns.tolist())
display(datos_comp_N505.head())

print("\nN1207 - Datos computacionales:")
display(datos_comp_N1207.head())

## 7. Correlación preliminar: Centralidad vs Tráfico

In [ ]:
# Promedio de tráfico por calle (sobre todos los instantes)
df['traffic_mean'] = df[traffic_cols].mean(axis=1)
df['traffic_normalized'] = df['traffic_mean'] / 255.0

# Scatter: Closeness Centrality vs Tráfico promedio
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (datos_comp, label) in zip(axes, [(datos_comp_N505, 'N505'), (datos_comp_N1207, 'N1207')]):
    cc = datos_comp['DiCC']
    cc_norm = (cc - cc.min()) / (cc.max() - cc.min())
    
    traffic_means = []
    for c in cc.index:
        if c < len(df):
            traffic_means.append(df.loc[c, 'traffic_normalized'])
        else:
            traffic_means.append(0)
    
    ax.scatter(cc_norm[:len(traffic_means)], traffic_means[:len(cc_norm)], 
              alpha=0.5, edgecolors='black', facecolors='white', s=40)
    ax.set_xlabel('Closeness Centrality (normalizado)')
    ax.set_ylabel('Tráfico promedio (normalizado)')
    ax.set_title(f'{label} - DiCC vs Tráfico')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('results/figures/preliminar_cc_vs_trafico.png', dpi=200, bbox_inches='tight')
plt.show()

## 8. Resumen del procesamiento

In [ ]:
# Verificar archivos generados
combinaciones_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.csv')])
print(f"Archivos de combinaciones generados: {len(combinaciones_files)}")
for f in combinaciones_files[:6]:
    print(f"  {f}")
if len(combinaciones_files) > 6:
    print(f"  ... y {len(combinaciones_files) - 6} más")

print("\n" + "=" * 50)
print("ARCHIVOS GENERADOS:")
print("=" * 50)
print("data/processed/combinaciones/traffic_{mean,max}_N{505,1207}_r{0-3}s{3,6,10}.csv")
print("=" * 50)